In [1]:
import pymongo
import time
from datetime import datetime

def prueba_consistencia_simple():
    """Prueba de consistencia simplificada"""
    print("🔍 PRUEBA DE CONSISTENCIA SIMPLIFICADA")
    print("="*50)
    
    # Conectar a los 3 nodos
    nodos = {}
    for puerto in [30001, 30002, 30003]:
        try:
            client = pymongo.MongoClient(f"mongodb://localhost:{puerto}/?directConnection=true", 
                                       serverSelectionTimeoutMS=3000)
            client.admin.command("ping")
            nodos[puerto] = client
            print(f"✅ Conectado a nodo {puerto}")
        except:
            print(f"❌ No se pudo conectar a nodo {puerto}")
    
    if len(nodos) < 2:
        print("❌ Necesitamos al menos 2 nodos activos")
        return False
    
    # 1. Insertar datos en el primer nodo disponible
    primer_nodo = list(nodos.values())[0]
    collection = primer_nodo["Politica"]["TestConsistencia"]
    
    # Limpiar y crear documento de prueba
    collection.delete_many({})
    doc_id = f"test_{int(time.time())}"
    
    collection.insert_one({
        "_id": doc_id,
        "mensaje": "Prueba de consistencia",
        "timestamp": datetime.now().isoformat()
    })
    
    print(f"📤 Insertado documento: {doc_id}")
    
    # 2. Esperar replicación
    time.sleep(3)
    
    # 3. Verificar en todos los nodos
    print("\n🔄 Verificando en todos los nodos:")
    consistente = True
    
    for puerto, client in nodos.items():
        test_collection = client["Politica"]["TestConsistencia"]
        doc = test_collection.find_one({"_id": doc_id})
        
        if doc:
            print(f"   ✅ Nodo {puerto}: Documento encontrado")
        else:
            print(f"   ❌ Nodo {puerto}: Documento NO encontrado")
            consistente = False
    
    # 4. Resultado
    if consistente:
        print("\n🎯 RESULTADO: ✅ DATOS CONSISTENTES")
    else:
        print("\n🎯 RESULTADO: ❌ DATOS INCONSISTENTES")
    
    # Limpiar
    collection.delete_many({})
    for client in nodos.values():
        client.close()
    
    return consistente

# Ejecutar prueba
prueba_consistencia_simple()

🔍 PRUEBA DE CONSISTENCIA SIMPLIFICADA
✅ Conectado a nodo 30001
✅ Conectado a nodo 30002
✅ Conectado a nodo 30003
📤 Insertado documento: test_1763350785

🔄 Verificando en todos los nodos:
   ✅ Nodo 30001: Documento encontrado
   ✅ Nodo 30002: Documento encontrado
   ✅ Nodo 30003: Documento encontrado

🎯 RESULTADO: ✅ DATOS CONSISTENTES


True

In [2]:
import subprocess

def prueba_alta_disponibilidad_simple():
    """Prueba de alta disponibilidad simplificada"""
    print("🚀 PRUEBA DE ALTA DISPONIBILIDAD SIMPLIFICADA")
    print("="*50)
    
    # 1. Conectar al replica set
    try:
        client = pymongo.MongoClient(
            "mongodb://localhost:30001,localhost:30002,localhost:30003/?replicaSet=my-replicaset",
            serverSelectionTimeoutMS=10000
        )
        client.admin.command("ping")
        print("✅ Conectado al replica set")
    except Exception as e:
        print(f"❌ Error inicial: {e}")
        return False
    
    # 2. Realizar operación inicial
    collection = client["Politica"]["TestDisponibilidad"]
    collection.delete_many({})
    
    doc_id = "test_disponibilidad"
    collection.insert_one({"_id": doc_id, "estado": "inicial"})
    print("✅ Operación inicial exitosa")
    
    # 3. Simular fallo del nodo principal
    print("\n🛑 Simulando fallo del nodo mongo1...")
    try:
        subprocess.run(["docker", "stop", "mongo1"], capture_output=True, timeout=10)
        print("✅ Nodo mongo1 detenido")
    except Exception as e:
        print(f"⚠️ Error al detener nodo: {e}")
    
    # 4. Esperar failover
    print("⏱️ Esperando failover automático...")
    time.sleep(10)
    
    # 5. Probar operaciones después del fallo
    try:
        # Intentar operación con el mismo cliente
        collection.update_one({"_id": doc_id}, {"$set": {"estado": "post_fallo"}})
        doc = collection.find_one({"_id": doc_id})
        
        if doc and doc["estado"] == "post_fallo":
            print("✅ Sistema sigue operativo después del fallo")
            alta_disponibilidad = True
        else:
            print("❌ Sistema no operativo")
            alta_disponibilidad = False
            
    except Exception as e:
        print(f"❌ Error después del fallo: {e}")
        alta_disponibilidad = False
    
    # 6. Restaurar nodo
    print("\n🔧 Restaurando nodo mongo1...")
    try:
        subprocess.run(["docker", "start", "mongo1"], capture_output=True, timeout=15)
        print("✅ Nodo mongo1 restaurado")
        time.sleep(5)
    except Exception as e:
        print(f"⚠️ Error al restaurar: {e}")
    
    # 7. Resultado
    if alta_disponibilidad:
        print("\n🎯 RESULTADO: ✅ ALTA DISPONIBILIDAD DEMOSTRADA")
    else:
        print("\n🎯 RESULTADO: ❌ FALLA EN ALTA DISPONIBILIDAD")
    
    # Limpiar
    try:
        collection.delete_many({})
        client.close()
    except:
        pass
    
    return alta_disponibilidad

# Ejecutar prueba
prueba_alta_disponibilidad_simple()

🚀 PRUEBA DE ALTA DISPONIBILIDAD SIMPLIFICADA
✅ Conectado al replica set
✅ Operación inicial exitosa

🛑 Simulando fallo del nodo mongo1...
✅ Nodo mongo1 detenido
⏱️ Esperando failover automático...
✅ Sistema sigue operativo después del fallo

🔧 Restaurando nodo mongo1...
✅ Nodo mongo1 restaurado

🎯 RESULTADO: ✅ ALTA DISPONIBILIDAD DEMOSTRADA


True